# Feature Engineering & Validation Pipeline — López de Prado Framework

**System purpose:** a repeatable pipeline that takes raw OHLCV → engineered features →
statistically validated, non-redundant, regime-stable feature set, ready for a
supervised model or a rule-based scanner (e.g. QMIE).

**Architecture (5 stages, each a hard gate — nothing proceeds to the next stage on failure):**

| Stage | Question answered | Failure mode it prevents |
|---|---|---|
| 0. Data integrity | Is the series point-in-time correct? | Look-ahead bias, survivorship bias |
| 1. Stationarity + Labeling | Is the target defined without leakage? | Fixed-horizon label leakage, non-stationary features breaking ML |
| 2. Feature engineering | What signal families do we compute? | Redundant single-indicator features |
| 3. Feature evaluation | Does a feature earn its place? | Overfitting via feature-count inflation |
| 4. Production gate | Does the surviving set survive OOS + regime shift? | Backtest-to-live decay (the #1 killer of retail systematic strategies) |

**Non-obvious framing:** most retail feature pipelines fail at Stage 0 and Stage 4, not at
feature engineering. You can have perfect features and still lose money if the labeling
scheme leaks future information (Stage 1) or if you never test for the Probability of
Backtest Overfitting (Stage 4). This notebook weights those two stages more heavily than
the feature functions themselves, on purpose.


In [ ]:

import numpy as np
import pandas as pd
from dataclasses import dataclass
from typing import List, Tuple, Optional
import warnings
warnings.filterwarnings("ignore")

RNG = np.random.default_rng(42)
pd.set_option("display.width", 140)


## Stage 0 — Synthetic OHLCV (swap for your real feed)

In production this cell is replaced by your point-in-time data loader. The **only**
requirement this pipeline imposes on that loader: **no field in the returned frame may
be computed using information not available at bar close `t`.** This sounds obvious and
is violated constantly — e.g. daily bars built from a vendor's "adjusted close" that
bakes in a future split/dividend, or a resampled higher-timeframe bar whose close is
still open at the time you'd have traded off it.

**Leverage point:** build a single `assert_no_lookahead(df)` gate once, run it in CI on
every data pull. Cheaper than finding the bug after 3 months of live P&L divergence.


In [ ]:

def make_synthetic_ohlcv(n=5000, regime_len=500, seed=42):
    '''Regime-switching synthetic OHLCV: alternates trend / mean-reversion / chop.'''
    rng = np.random.default_rng(seed)
    price = 100.0
    closes, regimes = [], []
    regime_cycle = ["trend_up", "trend_down", "meanrev", "chop"]
    for i in range(n):
        regime = regime_cycle[(i // regime_len) % len(regime_cycle)]
        if regime == "trend_up":
            drift, vol = 0.0006, 0.006
        elif regime == "trend_down":
            drift, vol = -0.0006, 0.006
        elif regime == "meanrev":
            drift, vol = -0.002 * np.tanh((price - 100) / 20), 0.004
        else:  # chop
            drift, vol = 0.0, 0.009
        price *= (1 + drift + rng.normal(0, vol))
        closes.append(price)
        regimes.append(regime)

    closes = np.array(closes)
    highs = closes * (1 + np.abs(rng.normal(0, 0.003, n)))
    lows = closes * (1 - np.abs(rng.normal(0, 0.003, n)))
    opens = np.roll(closes, 1); opens[0] = closes[0]
    volume = rng.lognormal(mean=9, sigma=0.5, size=n) * (1 + 0.5 * np.abs(np.diff(closes, prepend=closes[0])) / closes)

    idx = pd.date_range("2018-01-01", periods=n, freq="1h")
    df = pd.DataFrame({"open": opens, "high": highs, "low": lows, "close": closes,
                        "volume": volume, "true_regime": regimes}, index=idx)
    return df

def assert_no_lookahead(df: pd.DataFrame, feature_cols: List[str]):
    '''Stage-0 gate: every feature at t must be computable from data <= t.
    In this synthetic context we can't detect leakage automatically (no future
    join to check against) — this is a placeholder for the REAL gate you must
    write against your actual data-join logic: assert feature timestamp t was
    generated using only rows with index <= t at the time of feature computation,
    not at analysis time. Wire this into your feature store's build step, not here.
    '''
    assert df.index.is_monotonic_increasing, "index not sorted — resample/join bug upstream"
    assert not df[feature_cols].isna().all(axis=None), "all-NaN feature block — join bug"
    return True

ohlcv = make_synthetic_ohlcv()
ohlcv.head()


## Stage 1a — Fractional Differentiation (López de Prado, *AFML* Ch. 5)

Raw prices are non-stationary (ADF fails); naive `diff()` (returns) is stationary but
destroys long-memory / trend information the model needs. Fractional differentiation
finds the **minimum differencing order `d`** that achieves stationarity while retaining
maximum memory. This is the single highest-leverage fix for feature pipelines that feed
an ML model — most retail pipelines skip it and either (a) use raw price → model overfits
non-stationary drift, or (b) use simple returns → model loses all trend memory.

If you're feeding a rule-based scanner (not an ML model) this stage is optional — but
even then, run the ADF test as a **diagnostic** on your trend/momentum features so you
know which ones are stationary and which need regime-conditioning.


In [ ]:

def get_weights_ffd(d: float, thresh: float = 1e-5, max_size: int = 10000) -> np.ndarray:
    '''Fixed-width window fractional-diff weights (AFML snippet 5.3).'''
    w = [1.0]
    k = 1
    while True:
        w_ = -w[-1] / k * (d - k + 1)
        if abs(w_) < thresh or k > max_size:
            break
        w.append(w_)
        k += 1
    return np.array(w[::-1]).reshape(-1, 1)

def frac_diff_ffd(series: pd.Series, d: float, thresh: float = 1e-5) -> pd.Series:
    '''Fixed-width-window fractional differentiation (AFML snippet 5.3, no leakage:
    each output at t only uses inputs <= t).'''
    w = get_weights_ffd(d, thresh)
    width = len(w) - 1
    out = pd.Series(index=series.index, dtype=float)
    s_log = np.log(series)
    for i in range(width, len(series)):
        window = s_log.iloc[i - width: i + 1].values.reshape(-1, 1)
        out.iloc[i] = (w.T @ window).item()
    return out

def find_min_ffd_d(series: pd.Series, d_grid=np.linspace(0.1, 1.0, 19)) -> Tuple[float, pd.DataFrame]:
    '''Grid-search minimum d s.t. ADF rejects unit root at 95% -> stationary with max memory.'''
    from statsmodels.tsa.stattools import adfuller
    rows = []
    for d in d_grid:
        diffed = frac_diff_ffd(series, d).dropna()
        if len(diffed) < 50:
            continue
        stat, pval, *_ = adfuller(diffed, maxlag=1, regression="c", autolag=None)
        corr = np.corrcoef(diffed, series.loc[diffed.index])[0, 1]
        rows.append({"d": d, "adf_stat": stat, "p_value": pval, "corr_with_orig": corr})
    report = pd.DataFrame(rows)
    passing = report[report.p_value < 0.05]
    d_min = float(passing.d.min()) if len(passing) else float(d_grid[-1])
    return d_min, report

d_min, ffd_report = find_min_ffd_d(ohlcv["close"])
print(f"minimum stationarity-achieving d = {d_min:.2f}  (memory retained, corr={ffd_report.loc[ffd_report.d==d_min,'corr_with_orig'].values})")
ffd_report


## Stage 1b — Triple-Barrier Labeling + Meta-Labeling (AFML Ch. 3)

**This is where most homegrown pipelines leak.** A fixed-horizon label ("return over
next N bars") ignores path — a trade that hits your stop at bar 3 and then recovers by
bar N is labeled a winner. That's not what happened to your capital. The triple-barrier
method labels each observation by **which of three barriers is touched first**: profit-take,
stop-loss, or a vertical (time) barrier — matching how a real position actually exits.

**Meta-labeling** (secondary layer): train a model to predict *whether to take* a primary
signal, not *which direction*. This decouples "is there an edge in direction" (your
existing signal / feature set) from "is this specific instance high-confidence" (sizing).
For a system like QMIE with a deterministic primary scanner, meta-labeling is the highest-leverage
ML addition you can make without touching the core signal logic.


In [ ]:

def get_daily_vol(close: pd.Series, span: int = 100, lookback: int = 1) -> pd.Series:
    '''EWMA realized volatility of returns, used to set adaptive barrier width (AFML snippet 3.1).'''
    idx0 = close.index.searchsorted(close.index - pd.Timedelta(hours=lookback))
    idx0 = idx0[idx0 > 0]
    rets = close.iloc[idx0].values / close.loc[close.index[idx0 - 1]].values - 1
    vol = pd.Series(rets, index=close.index[idx0]).ewm(span=span).std()
    return vol.reindex(close.index).ffill()

def apply_triple_barrier(close: pd.Series, events: pd.DataFrame, pt_sl: Tuple[float, float],
                          vertical_days: int) -> pd.DataFrame:
    '''events: DataFrame indexed by signal timestamp, columns ['t1' (vertical barrier ts), 'trgt' (vol), 'side'].
    Returns first-touch timestamp and outcome per event (AFML snippet 3.2/3.3, vectorized-lite).'''
    out = events[["t1"]].copy()
    pt = pt_sl[0] * events["trgt"] if pt_sl[0] > 0 else pd.Series(index=events.index, dtype=float)
    sl = -pt_sl[1] * events["trgt"] if pt_sl[1] > 0 else pd.Series(index=events.index, dtype=float)

    for loc, t1 in events["t1"].fillna(close.index[-1]).items():
        path = close[loc:t1]
        if len(path) < 2:
            continue
        path_rets = (path / close[loc] - 1) * events.at[loc, "side"]
        touch_sl = path_rets[path_rets < sl.get(loc, -np.inf)].index.min() if pt_sl[1] > 0 else pd.NaT
        touch_pt = path_rets[path_rets > pt.get(loc, np.inf)].index.min() if pt_sl[0] > 0 else pd.NaT
        out.loc[loc, "sl"] = touch_sl
        out.loc[loc, "pt"] = touch_pt
    return out

def build_labels(close: pd.Series, side: pd.Series, pt_sl=(1.5, 1.0), vertical_bars=24,
                  vol_span=100) -> pd.DataFrame:
    '''Full triple-barrier labeling wrapper -> returns bin (-1/0/1), return, and holding period.'''
    trgt = get_daily_vol(close, span=vol_span)
    t1 = close.index.to_series().shift(-vertical_bars).reindex(close.index)
    events = pd.DataFrame({"t1": t1, "trgt": trgt, "side": side}).dropna(subset=["trgt", "side"])
    touches = apply_triple_barrier(close, events, pt_sl, vertical_bars)
    first_touch = touches[["sl", "pt", "t1"]].min(axis=1)

    labels = pd.DataFrame(index=events.index)
    labels["t1"] = first_touch
    labels["ret"] = close.reindex(first_touch.values).values / close.loc[events.index].values - 1
    labels["ret"] *= events["side"].values
    labels["bin"] = np.sign(labels["ret"])
    labels.loc[labels["ret"].abs() < 1e-6, "bin"] = 0
    labels["meta_label"] = (labels["bin"] > 0).astype(int)  # "was primary signal direction correct"
    return labels

# Example: primary signal = naive momentum sign (stand-in for your QMIE scanner output)
mom_signal = np.sign(ohlcv["close"].pct_change(12)).fillna(0).replace(0, 1)
labels = build_labels(ohlcv["close"], side=mom_signal, pt_sl=(1.5, 1.0), vertical_bars=24)
labels["bin"].value_counts(normalize=True)


## Stage 1c — Sample Uniqueness & Weighting (AFML Ch. 4)

Triple-barrier labels **overlap in time** (a label spanning bars 10→34 overlaps one
spanning bars 15→40). Standard IID cross-validation assumptions are violated — the model
sees near-duplicate information across "independent" folds, inflating in-sample accuracy
and hiding overfitting until live. Two fixes, both mandatory if you're training any ML
model on triple-barrier labels: **(1)** average uniqueness weighting during training,
**(2)** purging + embargo during cross-validation (Stage 3).


In [ ]:

def get_concurrency(close_index: pd.DatetimeIndex, t1: pd.Series) -> pd.Series:
    '''Number of concurrently-open labels at each timestamp (AFML snippet 4.1).'''
    conc = pd.Series(0, index=close_index)
    for start, end in t1.items():
        if pd.isna(end):
            continue
        conc.loc[start:end] += 1
    return conc.replace(0, np.nan)

def get_avg_uniqueness(t1: pd.Series, conc: pd.Series) -> pd.Series:
    '''Average uniqueness of each label over its lifespan -> sample weight (AFML snippet 4.2).'''
    weights = pd.Series(index=t1.index, dtype=float)
    for start, end in t1.items():
        if pd.isna(end):
            continue
        span = conc.loc[start:end]
        weights[start] = (1.0 / span).mean() if len(span) else np.nan
    return weights

concurrency = get_concurrency(ohlcv.index, labels["t1"])
sample_weights = get_avg_uniqueness(labels["t1"], concurrency)
labels["sample_weight"] = sample_weights
print(f"mean average uniqueness: {sample_weights.mean():.3f}  "
      f"(1.0 = no overlap, closer to 0 = heavy overlap -> down-weight in training)")


## Stage 2 — Feature Engineering: Six Families

Each family below is a **module**, not a grab-bag of indicators — the point of grouping
this way is that Stage 3's redundancy check will show you *within-family* correlation is
expected and fine (they're measuring the same underlying construct from different angles,
which is exactly how you build a robust composite), while *cross-family* correlation
surviving to production is a red flag (two families claiming to measure different things
but actually encoding the same signal → wasted model capacity, and in a rule-based scanner,
double-counted confluence score — the exact bug already flagged in your ORBIS confluence
scoring work).


In [ ]:

def rolling_slope_r2(s: pd.Series, window: int) -> Tuple[pd.Series, pd.Series]:
    x = np.arange(window)
    x_mean = x.mean(); x_var = ((x - x_mean) ** 2).sum()
    def _fit(y):
        y_mean = y.mean()
        slope = ((x - x_mean) * (y - y_mean)).sum() / x_var
        yhat = y_mean + slope * (x - x_mean)
        ss_res = ((y - yhat) ** 2).sum(); ss_tot = ((y - y_mean) ** 2).sum()
        r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
        return slope, r2
    slopes = s.rolling(window).apply(lambda y: _fit(y)[0], raw=True)
    r2s = s.rolling(window).apply(lambda y: _fit(y)[1], raw=True)
    return slopes, r2s

def hurst_exponent(s: pd.Series, window: int, max_lag: int = 20) -> pd.Series:
    '''Rolling Hurst via R/S proxy: >0.5 trending, <0.5 mean-reverting, ~0.5 random walk.'''
    def _hurst(x):
        lags = range(2, max_lag)
        tau = [np.std(np.subtract(x[lag:], x[:-lag])) for lag in lags]
        tau = [t if t > 0 else 1e-8 for t in tau]
        poly = np.polyfit(np.log(list(lags)), np.log(tau), 1)
        return poly[0] * 2.0
    return s.rolling(window).apply(lambda y: _hurst(y.values), raw=False)

def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    close, high, low, vol = df["close"], df["high"], df["low"], df["volume"]
    f = pd.DataFrame(index=df.index)

    # --- Trend strength ---
    slope20, r2_20 = rolling_slope_r2(np.log(close), 20)
    f["trend_slope_20"] = slope20
    f["trend_r2_20"] = r2_20
    tr = pd.concat([(high - low), (high - close.shift()).abs(), (low - close.shift()).abs()], axis=1).max(axis=1)
    atr14 = tr.rolling(14).mean()
    plus_dm = (high.diff().clip(lower=0)).where(high.diff() > -low.diff(), 0.0)
    minus_dm = (-low.diff().clip(upper=0)).where(-low.diff() > high.diff(), 0.0)
    plus_di = 100 * (plus_dm.rolling(14).mean() / atr14)
    minus_di = 100 * (minus_dm.rolling(14).mean() / atr14)
    dx = 100 * (plus_di - minus_di).abs() / (plus_di + minus_di).replace(0, np.nan)
    f["adx_14"] = dx.rolling(14).mean()

    # --- Breakout structure ---
    donchian_high20 = high.rolling(20).max()
    donchian_low20 = low.rolling(20).min()
    f["breakout_dist_up"] = (close - donchian_high20.shift(1)) / atr14
    f["breakout_dist_down"] = (donchian_low20.shift(1) - close) / atr14
    f["range_compression"] = (donchian_high20 - donchian_low20) / atr14  # low value = coiled range

    # --- Momentum ---
    f["roc_12"] = close.pct_change(12)
    f["roc_48"] = close.pct_change(48)
    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = (-delta.clip(upper=0)).rolling(14).mean()
    f["rsi_14"] = 100 - 100 / (1 + gain / loss.replace(0, np.nan))
    ema12, ema26 = close.ewm(span=12).mean(), close.ewm(span=26).mean()
    f["macd_hist"] = (ema12 - ema26) - (ema12 - ema26).ewm(span=9).mean()

    # --- Volatility ---
    ret = close.pct_change()
    f["realized_vol_20"] = ret.rolling(20).std()
    f["parkinson_vol_20"] = np.sqrt((1 / (4 * np.log(2))) * (np.log(high / low) ** 2).rolling(20).mean())
    f["vol_of_vol_20"] = f["realized_vol_20"].rolling(20).std()
    f["atr_pct"] = atr14 / close

    # --- Volume ---
    f["volume_z_20"] = (vol - vol.rolling(20).mean()) / vol.rolling(20).std()
    obv = (np.sign(delta) * vol).fillna(0).cumsum()
    f["obv_slope_20"], _ = rolling_slope_r2(obv, 20)
    dollar_vol = (close * vol)
    f["amihud_illiq_20"] = (ret.abs() / dollar_vol.replace(0, np.nan)).rolling(20).mean() * 1e6

    # --- Regime ---
    f["hurst_100"] = hurst_exponent(np.log(close), 100)
    f["vol_regime_z"] = (f["realized_vol_20"] - f["realized_vol_20"].rolling(200).mean()) / f["realized_vol_20"].rolling(200).std()
    f["chop_score"] = 1 - f["trend_r2_20"]  # high = choppy/non-trending

    return f

features = engineer_features(ohlcv)
assert_no_lookahead(features, list(features.columns))
features.tail()


## Stage 3 — Feature Evaluation Pipeline

Four checks, in order. **A feature must pass all four to survive** — this is a hard gate,
not a scoring average, because a feature that's highly predictive but unstable across
regimes (or redundant with an existing feature) is worse than no feature: it will look
great in-sample and then either flip sign live or add zero marginal information while
consuming model capacity / overfitting risk budget.

1. **Predictive power** — MDA (Mean Decrease Accuracy) under Purged K-Fold CV, not
   naive correlation-with-target (correlation ignores the label's path-dependent, path-barrier
   structure and non-linear interactions).
2. **Redundancy** — hierarchical correlation clustering; keep the best performer per cluster.
3. **Regime stability** — recompute MDA per regime slice; a feature whose importance
   flips sign or collapses in one regime is a landmine for a system that trades all regimes.
4. **Out-of-sample** — Combinatorial Purged CV (CPCV) + Deflated Sharpe Ratio at the
   **strategy** level, not just feature level — this is the check most pipelines skip
   entirely and it's the one that actually predicts live decay.


In [ ]:

class PurgedKFold:
    '''K-Fold CV with purging (drop train samples whose label window overlaps a test
    sample's window) and embargo (drop a buffer of train samples immediately after each
    test fold) — AFML Ch. 7. Prevents the single most common CV leak in financial ML.'''
    def __init__(self, n_splits: int, t1: pd.Series, embargo_pct: float = 0.01):
        self.n_splits = n_splits
        self.t1 = t1
        self.embargo_pct = embargo_pct

    def split(self, X: pd.DataFrame):
        idx = X.index
        n = len(idx)
        fold_bounds = [(i * n // self.n_splits, (i + 1) * n // self.n_splits) for i in range(self.n_splits)]
        embargo = int(n * self.embargo_pct)
        for start, end in fold_bounds:
            test_idx = idx[start:end]
            test_t1_max = self.t1.loc[test_idx].max()
            test_start = idx[start]
            train_mask = pd.Series(True, index=idx)
            # purge: drop train obs whose (t, t1) window overlaps test window
            overlap = (self.t1.reindex(idx) >= test_start) & (idx <= test_t1_max)
            train_mask &= ~overlap
            # embargo: drop `embargo` obs immediately after test fold end
            embargo_end = min(end + embargo, n)
            train_mask.iloc[end:embargo_end] = False
            train_mask.iloc[start:end] = False
            yield idx[train_mask], test_idx

def mda_feature_importance(X: pd.DataFrame, y: pd.Series, t1: pd.Series, sample_w: pd.Series,
                            n_splits: int = 5, n_estimators: int = 200) -> pd.DataFrame:
    '''Mean Decrease Accuracy under Purged K-Fold: shuffle one feature at a time on the
    TEST fold, measure accuracy drop. More robust to substitution effects than MDI
    (impurity-based importance), which inflates correlated features (AFML Ch. 8).'''
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.metrics import accuracy_score

    Xc = X.dropna()
    common = Xc.index.intersection(y.index).intersection(sample_w.dropna().index)
    Xc, yc, wc, t1c = X.loc[common], y.loc[common], sample_w.loc[common], t1.loc[common]

    pkf = PurgedKFold(n_splits, t1c)
    scores_base, scores_perm = {c: [] for c in Xc.columns}, {c: [] for c in Xc.columns}

    for train_idx, test_idx in pkf.split(Xc):
        if len(train_idx) < 50 or len(test_idx) < 10:
            continue
        clf = RandomForestClassifier(n_estimators=n_estimators, max_depth=5, n_jobs=-1, random_state=42)
        clf.fit(Xc.loc[train_idx], yc.loc[train_idx], sample_weight=wc.loc[train_idx])
        base_acc = accuracy_score(yc.loc[test_idx], clf.predict(Xc.loc[test_idx]))
        for col in Xc.columns:
            X_perm = Xc.loc[test_idx].copy()
            X_perm[col] = RNG.permutation(X_perm[col].values)
            perm_acc = accuracy_score(yc.loc[test_idx], clf.predict(X_perm))
            scores_base[col].append(base_acc)
            scores_perm[col].append(perm_acc)

    report = pd.DataFrame({
        "mda": {c: np.mean(scores_base[c]) - np.mean(scores_perm[c]) for c in Xc.columns}
    }).sort_values("mda", ascending=False)
    return report

common_idx = features.dropna().index.intersection(labels.index)
X = features.loc[common_idx]
y = labels.loc[common_idx, "meta_label"]
t1 = labels.loc[common_idx, "t1"]
w = labels.loc[common_idx, "sample_weight"].fillna(labels["sample_weight"].mean())

mda_report = mda_feature_importance(X, y, t1, w, n_splits=5)
mda_report


### 3.2 — Redundancy: hierarchical correlation clustering

Rather than a flat correlation-threshold cutoff (arbitrary, sensitive to which feature you
check first), cluster features hierarchically by `1 - |corr|` distance and keep only the
highest-MDA feature per cluster. This is AFML's "clustered feature importance" approach —
it correctly handles the case where 3 features are pairwise correlated at 0.6 (individually
below a naive 0.8 threshold) but jointly redundant.


In [ ]:

from scipy.cluster.hierarchy import linkage, fcluster
from scipy.spatial.distance import squareform

def cluster_redundant_features(X: pd.DataFrame, mda: pd.Series, corr_dist_thresh: float = 0.35):
    corr = X.corr().abs()
    dist_vals = (1 - corr).values.copy()
    np.fill_diagonal(dist_vals, 0)
    dist_vals = (dist_vals + dist_vals.T) / 2  # enforce exact symmetry (float rounding guard)
    condensed = squareform(dist_vals, checks=False)
    Z = linkage(condensed, method="average")
    cluster_ids = fcluster(Z, t=corr_dist_thresh, criterion="distance")
    cluster_map = pd.Series(cluster_ids, index=X.columns, name="cluster")

    keep = []
    for cid, group in cluster_map.groupby(cluster_map):
        members = group.index
        best = mda.loc[members].idxmax()
        keep.append(best)
    return keep, cluster_map

survivors_stage2, cluster_map = cluster_redundant_features(X, mda_report["mda"])
print(f"{len(survivors_stage2)} / {len(X.columns)} features survive redundancy clustering:")
print(survivors_stage2)
cluster_map.sort_values()


### 3.3 — Regime stability

Recompute MDA per `true_regime` slice (in production: per your realized regime classifier
output, e.g. the `hurst_100` / `vol_regime_z` features from Stage 2, or ORBIS's structure
engine classifier). A feature with high average MDA but high variance across regimes is
**more dangerous than a mediocre-but-stable feature** — it will get sized up by any
importance-weighted ensemble right before the regime turns against it.


In [ ]:

def regime_stability(X: pd.DataFrame, y: pd.Series, t1: pd.Series, sample_w: pd.Series,
                      regime_labels: pd.Series, feature_subset: List[str]) -> pd.DataFrame:
    rows = []
    for regime in regime_labels.unique():
        mask = regime_labels.reindex(X.index) == regime
        if mask.sum() < 200:
            continue
        Xr, yr, t1r, wr = X.loc[mask, feature_subset], y.loc[mask], t1.loc[mask], sample_w.loc[mask]
        rep = mda_feature_importance(Xr, yr, t1r, wr, n_splits=3, n_estimators=100)
        rep["regime"] = regime
        rows.append(rep.reset_index().rename(columns={"index": "feature"}))
    long = pd.concat(rows, ignore_index=True)
    pivot = long.pivot(index="feature", columns="regime", values="mda")
    pivot["mean"] = pivot.mean(axis=1)
    pivot["std"] = pivot.drop(columns="mean").std(axis=1)
    pivot["stability_ratio"] = pivot["mean"] / pivot["std"].replace(0, np.nan)
    return pivot.sort_values("stability_ratio", ascending=False)

regime_series = ohlcv["true_regime"].loc[common_idx]
stability_report = regime_stability(X, y, t1, w, regime_series, survivors_stage2)
stability_report


### 3.4 — Out-of-sample: CPCV + Deflated Sharpe Ratio

Run at the **strategy** level using only the surviving feature set: Combinatorial Purged
Cross-Validation generates many train/test path combinations (not just one walk-forward
path), giving a distribution of Sharpe ratios rather than a single lucky/unlucky number.
The Deflated Sharpe Ratio then corrects that distribution's best result for the number of
trials run (multiple testing) and for non-normal return skew/kurtosis — directly answering
"is this Sharpe likely a fluke given how many feature/parameter combinations I tried."

**This is the gate your memory shows you already treat as load-bearing** (CPCV + Deflated
Sharpe as a hard gate before UI/alerts/execution code) — wiring it here closes the loop
from feature selection through to the same statistical bar the rest of the system uses.


In [ ]:

from itertools import combinations
from scipy.stats import norm

def cpcv_paths(n_groups: int, k_test_groups: int) -> List[Tuple[Tuple[int, ...], Tuple[int, ...]]]:
    '''AFML Ch. 12: partition data into n_groups blocks, enumerate all C(n_groups, k_test)
    combinations as test sets -> N paths, each an unbiased backtest with proper purging.'''
    groups = list(range(n_groups))
    paths = []
    for test_groups in combinations(groups, k_test_groups):
        train_groups = tuple(g for g in groups if g not in test_groups)
        paths.append((train_groups, test_groups))
    return paths

def run_cpcv_backtest(X: pd.DataFrame, y: pd.Series, ret: pd.Series, t1: pd.Series,
                       sample_w: pd.Series, n_groups: int = 6, k_test: int = 2) -> pd.Series:
    '''Returns one realized strategy return series (using OOS predictions stitched
    across all CPCV paths) per path -> distribution of path Sharpe ratios.'''
    from sklearn.ensemble import RandomForestClassifier
    n = len(X)
    bounds = [(i * n // n_groups, (i + 1) * n // n_groups) for i in range(n_groups)]
    idx = X.index
    paths = cpcv_paths(n_groups, k_test)
    sharpes = []
    for train_groups, test_groups in paths:
        train_idx = pd.Index([]); test_idx = pd.Index([])
        for g in train_groups:
            s, e = bounds[g]; train_idx = train_idx.append(idx[s:e])
        for g in test_groups:
            s, e = bounds[g]; test_idx = test_idx.append(idx[s:e])
        # purge overlap between train and test label windows
        test_t1_max = t1.loc[test_idx].max()
        overlap = (t1.reindex(train_idx) >= test_idx.min()) & (train_idx <= test_t1_max)
        train_idx = train_idx[~overlap]
        if len(train_idx) < 100 or len(test_idx) < 20:
            continue
        clf = RandomForestClassifier(n_estimators=150, max_depth=5, n_jobs=-1, random_state=1)
        clf.fit(X.loc[train_idx], y.loc[train_idx], sample_weight=sample_w.loc[train_idx].fillna(sample_w.mean()))
        pred = clf.predict(X.loc[test_idx])
        strat_ret = ret.loc[test_idx] * pred  # meta-label gates the trade (0/1)
        if strat_ret.std() > 0:
            sharpes.append(strat_ret.mean() / strat_ret.std() * np.sqrt(252 * 24))  # hourly bars annualized
    return pd.Series(sharpes, name="path_sharpe")

def deflated_sharpe_ratio(sharpes: pd.Series, n_trials: int, skew: float = 0.0, kurt: float = 3.0) -> dict:
    '''AFML Ch. 14 (Bailey & Lopez de Prado 2014). Corrects best-observed Sharpe for
    (a) selection bias under multiple trials and (b) non-normal return distribution.'''
    sr = sharpes.max()
    sr_std = sharpes.std()
    n = len(sharpes)
    euler_gamma = 0.5772156649
    max_z = (1 - euler_gamma) * norm.ppf(1 - 1.0 / n_trials) + euler_gamma * norm.ppf(1 - 1.0 / (n_trials * np.e))
    sr0 = sr_std * max_z  # expected max Sharpe under null of n_trials independent zero-Sharpe strategies
    denom = np.sqrt(1 - skew * sr + (kurt - 1) / 4 * sr ** 2)
    dsr_stat = (sr - sr0) / (sr_std * denom) if sr_std > 0 else np.nan
    return {"best_sharpe": sr, "expected_max_sharpe_under_null": sr0,
            "deflated_sharpe_stat": dsr_stat, "deflated_sharpe_pvalue": 1 - norm.cdf(dsr_stat)}

ret_series = ohlcv["close"].pct_change().loc[common_idx]
path_sharpes = run_cpcv_backtest(X[survivors_stage2], y, ret_series, t1, w, n_groups=6, k_test=2)
dsr = deflated_sharpe_ratio(path_sharpes, n_trials=len(survivors_stage2) * 4)  # trials ~ feature combos tried
print(f"CPCV paths run: {len(path_sharpes)}")
print(f"Path Sharpe distribution: mean={path_sharpes.mean():.2f}  std={path_sharpes.std():.2f}  min={path_sharpes.min():.2f}")
print(dsr)


## Stage 4 — Final Selection Decision

Mechanical, not discretionary: a feature survives only if it clears **all** of MDA > 0,
redundancy-cluster survivor, and stability-ratio above a floor. Log every drop with a
reason — this log is your defense against re-litigating the same feature six months later.


In [ ]:

def select_final_features(mda: pd.DataFrame, redundancy_survivors: List[str],
                           stability: pd.DataFrame, stability_floor: float = 0.5) -> pd.DataFrame:
    rows = []
    for feat in mda.index:
        reasons = []
        passed = True
        if mda.loc[feat, "mda"] <= 0:
            passed = False; reasons.append("MDA <= 0 (no predictive power beyond noise)")
        if feat not in redundancy_survivors:
            passed = False; reasons.append("dropped in redundancy clustering (better proxy exists in its cluster)")
        if feat in stability.index and (stability.loc[feat, "stability_ratio"] < stability_floor or pd.isna(stability.loc[feat, "stability_ratio"])):
            passed = False; reasons.append(f"stability_ratio below floor ({stability_floor}) — flips sign across regimes")
        rows.append({"feature": feat, "survives": passed, "reasons": "; ".join(reasons) if reasons else "PASS"})
    return pd.DataFrame(rows).set_index("feature")

decision_log = select_final_features(mda_report, survivors_stage2, stability_report)
final_features = decision_log[decision_log.survives].index.tolist()
print(f"FINAL SET ({len(final_features)}/{len(mda_report)}): {final_features}")
decision_log


## Production Deployment Notes

**Infra (this pipeline as a scheduled job, not a notebook, in production):**
- Feature computation → a feature store (even a simple partitioned Parquet-on-disk /
  Timescale hypertable is enough at single-desk scale) written by a versioned, tested
  `features.py`, not this notebook. The notebook is for *research iteration*; production
  needs deterministic, replayable feature generation with a git-committed hash per feature
  version so you can trace any live signal back to the exact code that produced it.
- Re-run Stage 3 (feature evaluation) on a **rolling schedule** (monthly, or triggered by
  a live/backtest Sharpe divergence alert) — feature importance is not static; a feature
  that passes today can decay as the market microstructure it exploits gets arbitraged out.
- CPCV + DSR at Stage 3.4 is compute-heavy (`C(n_groups, k_test)` RF fits). Cache trained
  models per path; only re-run when the underlying feature set changes, not on every data refresh.

**Monitoring / drift:**
- Track live MDA-equivalent via a rolling out-of-time holdout, not just backtest MDA.
  Divergence between backtest-period MDA and rolling live MDA for the same feature is your
  earliest drift signal — earlier than P&L decay, which lags by definition.
- Alert on `stability_ratio` recomputed live per realized regime — if a feature's regime-conditional
  behavior starts resembling a regime it historically failed in, that's a pre-emptive kill signal.

**Failure modes specific to this pipeline (not generic ML advice):**
1. **Synthetic-to-real gap**: the regime-switching synthetic data here has clean regime
   boundaries; real markets have ambiguous, gradual regime transitions. Stage 3.3's
   regime-stability check is only as good as your regime labels — if you're using `true_regime`
   in production (you can't, it's synthetic), substitute your actual classifier output and
   expect stability ratios to be noisier and lower across the board.
2. **MDA under-detects features that matter only in tails**: RF-based MDA with balanced
   accuracy scoring will systematically undervalue a feature that only fires (and matters)
   during rare breakout events — if your model's real edge is tail-event capture (plausible
   given your breakout-structure family), pair MDA with a tail-conditional score (e.g. MDA
   computed only on the top/bottom decile of `|ret|`) before dropping a low-average-MDA feature.
3. **CPCV path count vs. compute budget**: `C(6,2) = 15` paths here; at full feature-family
   scale with hyperparameter search this becomes the actual bottleneck of the whole pipeline —
   budget for it explicitly rather than discovering it live during a research sprint.
4. **Meta-labeling assumes the primary signal has *some* edge**: if the primary scanner's
   direction call is worse than random, meta-labeling optimizes "when to skip a coin-flip"
   which caps achievable Sharpe far below what feature quality alone would suggest — validate
   primary-signal edge (bin distribution in Stage 1b, not 50/50) before investing further
   in feature engineering around it.

**Monetization angle, if this becomes a component (not a strategy) for sale:** the
defensible IP here is not any single feature — indicators are commoditized — it's the
**validation pipeline discipline** (CPCV + DSR + regime-stability as hard gates). Packaging
this as a feature-vetting-as-a-service layer (accepts a feature function, an OHLCV feed,
and a labeling scheme; returns a pass/fail decision log like the one above) is a more
defensible product than another signal service, because it's infrastructure other quants
need regardless of whose signals they're running.
